# ELM Validation: Benchmark Results

12 LLMs evaluated on 31 ELM-CPG pairs (15 valid, 16 invalid).

Frontier model results are **means over 5 independent trials** at temperature 0.1. Small model results are single runs (deterministic at T=0).

In [1]:
import pandas as pd, numpy as np, os, warnings
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore')

# --- Locate directories ---
MT_DIR = SR_DIR = None
for d in ['../results/multi_trial', 'results/multi_trial']:
    if os.path.isdir(d): MT_DIR = d; break
for d in ['../results', 'results']:
    if os.path.isdir(d): SR_DIR = d; break

FRONTIER = ['llama-3.3-70b', 'gpt-oss-20b', 'gpt-oss-120b', 'qwen3-32b']
SMALL = ['medgemma-4b', 'medgemma-1.5-4b', 'gemma-3-4b', 'phi-3-mini',
         'qwen-2.5-3b', 'llama-3.2-3b', 'qwen-2.5-1.5b', 'llama-3.2-1b']

def load_csv(path):
    df = pd.read_csv(path)
    for col in ['valid', 'correct', 'expected_valid']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower() == 'true'
    df['case'] = df['file'].str.replace('.json', '', regex=False)
    return df

# Multi-trial frontier data (standard prompt, 5 trials x 4 models)
mt_frames = []
for f in sorted(os.listdir(MT_DIR)):
    if f.startswith('results-') and f.endswith('.csv') and 'few-shot' not in f:
        mt_frames.append(load_csv(os.path.join(MT_DIR, f)))
mt_data = pd.concat(mt_frames, ignore_index=True)

# Single-run small model data
sr_frames = []
for f in sorted(os.listdir(SR_DIR)):
    if not f.startswith('results-') or not f.endswith('.csv'): continue
    path = os.path.join(SR_DIR, f)
    if not os.path.isfile(path): continue
    df = load_csv(path)
    if df['model'].iloc[0] in SMALL:
        sr_frames.append(df)
sr_data = pd.concat(sr_frames, ignore_index=True)

n_cases = mt_data['file'].nunique()
ref = mt_data[(mt_data['model'] == FRONTIER[0]) & (mt_data['trial'] == 1)]
n_valid = int(ref['expected_valid'].sum())
n_invalid = n_cases - n_valid
n_trials = mt_data['trial'].nunique()

print(f"Multi-trial: {mt_data['model'].nunique()} frontier models x {n_trials} trials x {n_cases} cases")
print(f"Single-run:  {sr_data['model'].nunique()} small models x {n_cases} cases")
print(f"Cases: {n_valid} valid, {n_invalid} invalid (base rate: {n_valid/n_cases:.1%})")

Multi-trial: 4 frontier models x 5 trials x 31 cases
Single-run:  8 small models x 31 cases
Cases: 15 valid, 16 invalid (base rate: 48.4%)


## Per-Model Accuracy (Multi-Trial Means for Frontier, Single Run for Small)

In [2]:
MODEL_ORDER = FRONTIER + SMALL
rows = []

# Frontier: per-trial metrics, then mean ± SD
for model in FRONTIER:
    mdf = mt_data[mt_data['model'] == model]
    per_trial = []
    for t in sorted(mdf['trial'].unique()):
        tdf = mdf[mdf['trial'] == t]
        n = len(tdf); k = int(tdf['correct'].sum())
        tp = int((tdf['valid'] & tdf['expected_valid']).sum())
        fp = int((tdf['valid'] & ~tdf['expected_valid']).sum())
        tn = int((~tdf['valid'] & ~tdf['expected_valid']).sum())
        fn = int((~tdf['valid'] & tdf['expected_valid']).sum())
        per_trial.append({
            'acc': k / n,
            'sens': tp / (tp + fn) if (tp + fn) else 0,
            'spec': tn / (tn + fp) if (tn + fp) else 0,
            'f1': 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0,
        })
    ptdf = pd.DataFrame(per_trial)
    rows.append({
        'Model': model, 'Trials': len(per_trial),
        'Accuracy': f"{ptdf['acc'].mean():.1%}",
        '±SD': f"{ptdf['acc'].std():.1%}",
        'Sens': f"{ptdf['sens'].mean():.2f}",
        'Spec': f"{ptdf['spec'].mean():.2f}",
        'F1': f"{ptdf['f1'].mean():.2f}",
    })

# Small: single run
for model in SMALL:
    mdf = sr_data[sr_data['model'] == model]
    if len(mdf) == 0: continue
    n = len(mdf); k = int(mdf['correct'].sum()); acc = k / n
    tp = int((mdf['valid'] & mdf['expected_valid']).sum())
    fp = int((mdf['valid'] & ~mdf['expected_valid']).sum())
    tn = int((~mdf['valid'] & ~mdf['expected_valid']).sum())
    fn = int((~mdf['valid'] & mdf['expected_valid']).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0
    spec = tn / (tn + fp) if (tn + fp) else 0
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0
    rows.append({
        'Model': model, 'Trials': 1,
        'Accuracy': f"{acc:.1%}",
        '±SD': '---',
        'Sens': f"{sens:.2f}",
        'Spec': f"{spec:.2f}",
        'F1': f"{f1:.2f}",
    })

pd.DataFrame(rows)

,Model,Trials,Accuracy,±SD,Sens,Spec,F1
0,llama-3.3-70b,5,89.0%,1.8%,0.92,0.86,0.89
1,gpt-oss-20b,5,88.4%,1.8%,0.93,0.84,0.89
2,gpt-oss-120b,5,88.4%,1.8%,0.89,0.88,0.88
3,qwen3-32b,5,85.8%,1.8%,0.99,0.74,0.87
4,medgemma-4b,1,48.4%,---,1.00,0.00,0.65
5,medgemma-1.5-4b,1,48.4%,---,1.00,0.00,0.65
6,gemma-3-4b,1,48.4%,---,1.00,0.00,0.65
7,phi-3-mini,1,54.8%,---,0.93,0.19,0.67
8,qwen-2.5-3b,1,61.3%,---,1.00,0.25,0.71
9,llama-3.2-3b,1,45.2%,---,0.87,0.06,0.60


## Fisher's Exact Test: Frontier vs Small

In [3]:
# Majority vote across 5 trials for frontier models (correct if >=3/5 trials correct)
frontier_mv = mt_data.groupby(['model', 'file']).agg(
    n_correct=('correct', 'sum'),
    expected_valid=('expected_valid', 'first')
).reset_index()
frontier_mv['correct'] = frontier_mv['n_correct'] >= 3

lc = int(frontier_mv['correct'].sum()); lt = len(frontier_mv)
sc = int(sr_data['correct'].sum()); st = len(sr_data)
odds, p = stats.fisher_exact([[lc, lt - lc], [sc, st - sc]])
print(f"Frontier (majority vote over {n_trials} trials): {lc}/{lt} ({lc/lt:.1%})")
print(f"Small (single run):    {sc}/{st} ({sc/st:.1%})")
print(f"Fisher exact: OR={odds:.2f}, p={p:.2e}")

Frontier (majority vote over 5 trials): 111/124 (89.5%)
Small (single run):    129/248 (52.0%)
Fisher exact: OR=7.88, p=1.13e-13


## McNemar Pairwise Tests (Bonferroni corrected)

In [4]:
from itertools import combinations

# Build unified per-case correct/incorrect: majority vote for frontier, single-run for small
frontier_mv['valid_pred'] = mt_data.groupby(['model', 'file'])['valid'].apply(
    lambda x: x.sum() >= 3).reset_index(drop=True) if False else None
combined = pd.concat([
    frontier_mv[['model', 'file', 'correct']],
    sr_data[['model', 'file', 'correct']]
], ignore_index=True)

MODEL_ORDER = FRONTIER + SMALL
functioning = [m for m in MODEL_ORDER if m in combined['model'].unique()]
pairs = list(combinations(functioning, 2))
alpha_corr = 0.05 / len(pairs)
print(f"{len(pairs)} pairs, Bonferroni alpha={alpha_corr:.5f}")
print(f"Frontier models use majority vote across {n_trials} trials\n")

sig_pairs = []
for m1, m2 in pairs:
    d1 = combined[combined['model'] == m1].set_index('file')['correct']
    d2 = combined[combined['model'] == m2].set_index('file')['correct']
    common = d1.index.intersection(d2.index)
    b = int((d1[common] & ~d2[common]).sum())
    c = int((~d1[common] & d2[common]).sum())
    nd = b + c
    p = 1.0 if nd == 0 else stats.binomtest(max(b, c), nd, 0.5).pvalue
    if p < 0.05:
        sig_pairs.append({
            'Pair': f'{m1} vs {m2}', 'b': b, 'c': c,
            'p': f'{p:.4f}', 'Sig': '**' if p < alpha_corr else '*'
        })

if sig_pairs:
    print("Pairs reaching uncorrected significance (p<0.05):")
    display(pd.DataFrame(sig_pairs))
else:
    print("No pairs reached p<0.05")

66 pairs, Bonferroni alpha=0.00076
Frontier models use majority vote across 5 trials

Pairs reaching uncorrected significance (p<0.05):


,Pair,b,c,p,Sig
0,llama-3.3-70b vs medgemma-4b,14,1,0.0010,*
1,llama-3.3-70b vs medgemma-1.5-4b,14,1,0.0010,*
2,llama-3.3-70b vs gemma-3-4b,14,1,0.0010,*
3,llama-3.3-70b vs phi-3-mini,12,1,0.0034,*
4,llama-3.3-70b vs qwen-2.5-3b,10,1,0.0117,*
5,llama-3.3-70b vs llama-3.2-3b,15,1,0.0005,**
6,llama-3.3-70b vs qwen-2.5-1.5b,14,2,0.0042,*
7,llama-3.3-70b vs llama-3.2-1b,11,1,0.0063,*
8,gpt-oss-20b vs medgemma-4b,14,0,0.0001,**
9,gpt-oss-20b vs medgemma-1.5-4b,14,0,0.0001,**
